# 달빛 윷마당 — Laya 파인튜닝
기존 3수 탐색 AI를 교사로 사용해 Laya가 윷놀이 선택 정책을 학습할 수 있는지 확인합니다. 이 단계의 목표는 교사 모방이며, 교사를 능가하려면 이후 더 깊은 탐색/MCTS 라벨이 필요합니다.

In [ ]:
!nvidia-smi
!python -m pip install -q 'laya==0.4.15' 'transformers==5.20.2' 'huggingface-hub==2.3.0' 'safetensors==0.8.0'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/dalbit-yut-laya

In [ ]:
!git clone -q https://github.com/zaruous/dalbit-yut.git /content/dalbit-yut
%cd /content/dalbit-yut
!npm install --package-lock=false --silent
!npm test

In [ ]:
!node scripts/generate-laya-training.cjs 6000 1200 /content/laya-data
!du -h /content/laya-data/*
!laya-train --data /content/laya-data/train.jsonl --eval /content/laya-data/eval.jsonl --base convaiinnovations/laya --out /content/drive/MyDrive/dalbit-yut-laya/checkpoint --device cuda --loss soft-ce --shuffle-options --epochs 4 --micro-batch 1 --grad-accum 32 --max-len 1024 --head-max-len 512 --dry-run

In [ ]:
!laya-train --data /content/laya-data/train.jsonl --eval /content/laya-data/eval.jsonl --base convaiinnovations/laya --out /content/drive/MyDrive/dalbit-yut-laya/checkpoint --device cuda --loss soft-ce --shuffle-options --epochs 4 --micro-batch 1 --grad-accum 32 --max-len 1024 --head-max-len 512

In [ ]:
import os
os.environ.update({
    'LAYA_PYTHON': '/usr/bin/python3',
    'LAYA_MODEL': '/content/drive/MyDrive/dalbit-yut-laya/checkpoint',
    'LAYA_DEVICE': 'cuda',
    'LAYA_INPUT_VERSION': 'v2',
    'LAYA_MAX_LEN': '1024',
    'LAYA_HEAD_MAX_LEN': '512',
    'LAYA_DECISION_MS': '15000',
})
!node scripts/benchmark-laya.cjs 10 6000

검증 승률이 유의미하게 상승한 경우에만 25 seeds(100게임)로 확대하세요: `!node scripts/benchmark-laya.cjs 25 7000`. 체크포인트와 학습 보고서는 `/content/drive/MyDrive/dalbit-yut-laya/checkpoint`에 영구 저장됩니다.